# Stage 14 - more training data for the fine-tuned reranker

Design: `docs/stage14_data_scale.md`.

## Setup

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'
ACCOUNT_LABEL = 'A'          # run-log identifier
CLEAR_STALE_LOCK = False     # requires a stopped lock holder
MODE = 'fresh'               # fresh or resume

from google.colab import drive
drive.mount('/content/drive')

import os, shlex, subprocess, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'no config.py under {PROJECT_DIR!r} - the shared folder is not mounted at this path.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
GUARD = f' --account {ACCOUNT_LABEL}' + (' --clear-stale-lock' if CLEAR_STALE_LOCK else '')


def run(cmd):
    """Stream command output and raise on a nonzero exit."""
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed: {cmd}')

## Install dependencies

In [ ]:
run('pip install -q -r requirements.txt')

## Preflight

In [ ]:
run('python -u scripts/35_preflight_stage14.py --gpu')

## Mine the shards

In [ ]:
run('python -u scripts/32_build_stage14_data.py' + GUARD)

## Train the 4k and 10k arms

In [ ]:
run(f'python -u scripts/33_train_data_scale.py --mode {MODE}' + GUARD)

## Dev bench

In [ ]:
run('python -u scripts/34_eval_data_scale.py --dev' + GUARD)

## Stage 6 bench

In [ ]:
run('python -u scripts/34_eval_data_scale.py' + GUARD)

## Results

In [ ]:
import json, pathlib
from IPython.display import Markdown, display

latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
summary = latest / 'stage14_summary.md'
if summary.exists():
    display(Markdown(summary.read_text(encoding='utf-8')))
    print(json.dumps(json.loads((latest / 'stage14_verdict.json').read_text(encoding='utf-8')),
                     indent=2))
else:
    print('no Stage 14 summary - the evaluation did not finish.')